In [1]:
!pip install ipython==8.12.0

In [2]:
%load_ext autoreload
%autoreload 2

<a href="https://colab.research.google.com/github/47v0/cs4782-final-project/blob/main/03_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import sys, os

# Clone the repository if it doesn't exist
PARENT_DIR = '/content/gdrive/MyDrive/final_project'
REPO_DIR   = f'{PARENT_DIR}/cs4782-final-project'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/47v0/cs4782-final-project.git "{REPO_DIR}"
else:
    !cd "{REPO_DIR}" && git pull

# Verify
!ls "{REPO_DIR}"

# Add the repository root to sys.path
code_dir = os.getcwd()
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

import torch
import pandas as pd
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Already up to date.
code  data  LICENSE  plan.txt  poster  README.md  report  results
PyTorch: 2.10.0+cpu


In [30]:
from google.colab import drive
drive.mount('/content/gdrive')
base_dir = "/content/gdrive/MyDrive/final_project/cs4782-final-project/code"
sys.path.append(base_dir)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [28]:
print(os.getcwd())
print(os.listdir(".."))

from train import paper_cfg, Trainer, build_dataloaders, build_model

/content
['proc', 'lib32', 'etc', 'lib', 'srv', 'home', 'sys', 'boot', 'var', 'libx32', 'usr', 'sbin', 'root', 'mnt', 'tmp', 'run', 'media', 'dev', 'opt', 'bin', 'lib64', 'content', 'kaggle', '.dockerenv', 'datalab', 'tools', 'python-apt', 'python-apt.tar.xz']


In [ ]:
cfg = paper_cfg()

# experiment
cfg["dataset"]  = "ETTh1"  # ETTh1 ETTh2 ETTm1 ETTm2 Weather Traffic Electricity ILI
cfg["seq_len"]  = 336      # look-back window  (PatchTST/42 uses L=336)
cfg["pred_len"] = 96       # forecast horizon  {96, 192, 336, 720}

# training
cfg["batch_size"] = 128
cfg["lr"]         = 1e-4
cfg["epochs"]     = 100
cfg["patience"]   = 10    # early-stopping on val MSE

#output paths
cfg["checkpoint_dir"] = "../../results/checkpoints"
cfg["log_dir"]        = "../../results/logs"

print("Config:")
for k, v in cfg.items():
    print(f"  {k:22s} = {v}")

## SANITY CHECKS

In [33]:
train_loader, val_loader, test_loader = build_dataloaders(cfg)

x_batch, y_batch = next(iter(train_loader))
print(f"x shape: {tuple(x_batch.shape)}  ->  expected (batch, M, {cfg['seq_len']})")
print(f"y shape: {tuple(y_batch.shape)}  ->  expected (batch, M, {cfg['pred_len']})")

x shape: (128, 7, 336)  ->  expected (batch, M, 336)
y shape: (128, 7, 96)  ->  expected (batch, M, 96)


In [34]:
model = build_model(cfg)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

with torch.no_grad():
    dummy = torch.randn(4, x_batch.shape[1], cfg["seq_len"])
    out   = model(dummy.to(cfg["device"]))
    print(f"Output shape: {tuple(out.shape)}  ->  expected (4, {x_batch.shape[1]}, {cfg['pred_len']})")

Total params: 7,024
Output shape: (4, 7, 96)  ->  expected (4, 7, 96)


## TRAINING

In [ ]:
trainer = Trainer(cfg)
results = trainer.fit()

In [ ]:
print("\nResults")
print(f"Test MSE : {results['test_mse']:.4f}")
print(f"Test MAE : {results['test_mae']:.4f}")
print(f"Best epoch: {results['best_epoch']}")
print(f"Paper target (ETTh1, T=96): MSE=0.375  MAE=0.399")

In [ ]:
import glob

log_path = os.path.join(
    cfg["log_dir"],
    f"train_{cfg['dataset']}_L{cfg['seq_len']}_T{cfg['pred_len']}.csv"
)
df = pd.read_csv(log_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric in zip(axes, ["mse", "mae"]):
    ax.plot(df["epoch"], df[f"train_{metric}"], label="train")
    ax.plot(df["epoch"], df[f"val_{metric}"],   label="val")
    ax.axvline(results["best_epoch"], color="red", ls="--",
               label=f"best (ep {results['best_epoch']})")
    ax.set_xlabel("Epoch"); ax.set_ylabel(metric.upper()); ax.set_title(metric.upper())
    ax.legend()

plt.suptitle(f"PatchTST — {cfg['dataset']} L={cfg['seq_len']} T={cfg['pred_len']}")
plt.tight_layout(); plt.show()

In [ ]:
# uncomment to run the full ETTh1 sweep (each are like ~10 min on T4)

# PAPER = {96: (0.375, 0.399), 192: (0.414, 0.421),
#          336: (0.431, 0.436), 720: (0.449, 0.466)}

# rows = []
# for T in [96, 192, 336, 720]:
#     print(f"\n{'='*50}\nT = {T}\n{'='*50}")
#     run_cfg = {**cfg, "pred_len": T}
#     res = Trainer(run_cfg).fit()
#     p_mse, p_mae = PAPER[T]
#     rows.append({"T": T,
#                  "Our MSE": round(res["test_mse"],4), "Paper MSE": p_mse,
#                  "Our MAE": round(res["test_mae"],4), "Paper MAE": p_mae})

# pd.DataFrame(rows).set_index("T")